# 01 – Data Quality

**Purpose:** Check whether the data can be trusted before modelling, and record a decision for every issue.

**Inputs:** tables from 00
**Outputs:** DQ scorecard (8 checks)

**Key results:**
- No duplicate keys; no orphan items or stores
- Promo flag unknown before Apr 2014 (one clean block); out of model scope
- Negative sales 0.006% of rows → clipped to 0 in a clean column
- Oil missing on 31% of days → forward-fill (no leakage)
- Holidays not unique by date → collapsed to flags per date + location before joining
- Missing store-days are not treated as 0 sales or traffic

In [1]:
import duckdb
con = duckdb.connect('../data/favorita.duckdb')   # creates the file on first run
con.sql("SET memory_limit = '4GB'")
con.sql("SET preserve_insertion_order = false")
con.sql("SET temp_directory = '../data/tmp'")

## DQ Check 1 – Checking duplicates in the sales table

In [3]:
con.sql(""" DESCRIBE SELECT * FROM sales""").df()
            
            

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,store_nbr,INTEGER,YES,None,None,None
2,item_nbr,INTEGER,YES,None,None,None
3,unit_sales,DOUBLE,YES,None,None,None
4,is_on_promotion,BOOLEAN,YES,None,None,None


In [5]:
con.sql("""SELECT 
            date,
            store_nbr,
            item_nbr,
            COUNT(*)
            FROM sales
            GROUP BY date, store_nbr, item_nbr
            HAVING COUNT(*) > 1""").df()

,date,store_nbr,item_nbr,count_star()


## DQ Check 2 – Checking missing promotion flags in the sales table

In [6]:
con.sql(""" DESCRIBE SELECT * FROM sales""").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,store_nbr,INTEGER,YES,None,None,None
2,item_nbr,INTEGER,YES,None,None,None
3,unit_sales,DOUBLE,YES,None,None,None
4,is_on_promotion,BOOLEAN,YES,None,None,None


In [9]:
con.sql("""SELECT 
            COUNT(*) AS tot_rows,
            SUM(CASE WHEN is_on_promotion IS NULL THEN 1 ELSE 0 END) AS null_promo_rows,
            SUM(CASE WHEN is_on_promotion IS NULL THEN 1 ELSE 0 END)*100.0/COUNT(*) AS pct_on_promo_null,
            MIN(CASE WHEN is_on_promotion IS NULL THEN date END) AS earliest_null_promo,
            MAX(CASE WHEN is_on_promotion IS NULL THEN date END) AS latest_null_promo
            FROM sales""").df()
            

,tot_rows,null_promo_rows,pct_on_promo_null,earliest_null_promo,latest_null_promo
0,125497040,21657651.0,17.257499,2013-01-01,2014-03-31


**Around 17% item_store_date combinations don't have promotion flag 
The promotion flag is missing from 2013/01/01 till 2014/03/31
All combinations post 2014/03/31 have promotion flag unless date is missing in the table for that combination

** checking if any promotional flag was available before 2014/03/31

In [11]:
con.sql("""SELECT
            COUNT(*)  no_of_comb_on_promo
            FROM sales 
            WHERE date < '2014-04-01'
            AND is_on_promotion	IS NOT NULL""").df()

,no_of_comb_on_promo
0,0


**Finding:** The promotion flag is unknown (NULL) for every row from 2013-01-01 to 2014-03-31 (21.7M rows, 17.3%) and known for every row from 2014-04-01 on. It is one clean block: the flag was not recorded before April 2014. **Decision:** no impact, because model scope starts in 2016.

## DQ Check 3 – Checking negative sales (returns) in the sales table

In [14]:
con.sql("""SELECT 
            COUNT(*) AS total_rows,
            SUM(CASE WHEN unit_sales<0 THEN 1 ELSE 0 END) AS negative_sales_rows,
            SUM(CASE WHEN unit_sales<0 THEN 1 ELSE 0 END)*100.0/COUNT(*) AS pct_neg_sales_rows,
            ABS(SUM(CASE WHEN unit_sales<0 THEN unit_sales ELSE 0 END)) AS sum_net_negative_sales,
            SUM(CASE WHEN unit_sales>0 THEN unit_sales ELSE 0 END) AS sum_net_positive_sales,
            ABS(SUM(CASE WHEN unit_sales<0 THEN unit_sales ELSE 0 END))*100.0/SUM(CASE WHEN unit_sales>0 THEN unit_sales ELSE 0 END) AS pct_neg_vs_pos_sales
            FROM sales""").df()

,total_rows,negative_sales_rows,pct_neg_sales_rows,sum_net_negative_sales,sum_net_positive_sales,pct_neg_vs_pos_sales
0,125497040,7795.0,0.006211,143580.8275,1.073754e+09,0.013372


**Finding:** 7,795 rows (0.006% of 125.5M) have negative `unit_sales`, totalling 143,581 units against 1.07B net-positive units (0.013%). The data holds **net** units per item, store and day (sales minus returns), so these rows are only the days when returns exceeded sales. True returns can't be measured: returns inside positive days are invisible.

**Decision:** the issue is immaterial. Keep raw `unit_sales` unchanged and create `unit_sales_clean` with negatives clipped to 0 in the combined table. Imputing with an item-store mean was rejected: it invents demand, leaks future information, and is biased upward because zero-sales days are missing.

## DQ Check 4 – Checking sales records with no master data (orphan IDs)

In [22]:
con.sql("""SELECT 
            COUNT(DISTINCT s.item_nbr) AS orphan_item_id
            FROM sales s
            LEFT JOIN items i
                ON s.item_nbr = i.item_nbr
            WHERE i.item_nbr IS NULL""").df()

,orphan_item_id
0,0


In [21]:
con.sql("""SELECT 
            COUNT(DISTINCT s.store_nbr) AS orphan_store_id
            FROM sales s
            LEFT JOIN stores st
                ON s.store_nbr = st.store_nbr
            WHERE st.store_nbr IS NULL""").df()

,orphan_store_id
0,0


**Finding:** 0 orphan items and 0 orphan stores. Every `item_nbr` and `store_nbr` in `sales` exists in `items` and `stores`.

# ## DQ Check 5 – Checking gaps and missing prices in the oil table

In [27]:
con.sql(""" WITH oil_date_missing AS(
            SELECT 
            date,
            LAG(date, 1) OVER( ORDER BY date) AS last_row_date,
            CAST(date - LAG(date, 1) OVER( ORDER BY date) AS INTEGER) AS curr_prev_date_diff
            FROM oil)
            SELECT 
            SUM(curr_prev_date_diff-1) AS no_of_dates_missing
            FROM oil_date_missing
            WHERE curr_prev_date_diff>1
            """).df()

,no_of_dates_missing
0,486.0


** Alternate way

In [28]:
con.sql("""SELECT
            MAX(date)-MIN(date)+1 - COUNT(*) AS no_of_dates_missing
            FROM oil""").df()

,no_of_dates_missing
0,486


** dates exists but no price

In [29]:
con.sql("""DESCRIBE SELECT * FROM oil""").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,oil_price_wti,DOUBLE,YES,None,None,None


In [30]:
con.sql("""SELECT
            COUNT(*) AS dates_price_missing
            FROM oil
            WHERE oil_price_wti IS NULL""").df()

,dates_price_missing
0,43


**percentage of calendar days in the range would have no oil price for a daily join

In [35]:
con.sql("""SELECT 
            MAX(date)-Min(date)+1 - COUNT(*) AS oil_date_missing,
            SUM(CASE WHEN oil_price_wti IS NULL THEN 1 ELSE 0 END) AS dates_price_missing,
            (MAX(date)-Min(date)+1 - COUNT(*) + SUM(CASE WHEN oil_price_wti IS NULL THEN 1 ELSE 0 END))*100.0/(MAX(date)-Min(date)+1) AS pct_date_price_missing
            FROM oil""").df()

,oil_date_missing,dates_price_missing,pct_date_price_missing
0,486,43.0,31.044601


**Finding:** Over 2013-01-01 to 2017-08-31 (1,704 calendar days), 529 days (31%) have no oil price: 486 days have no row (weekends and market holidays) and 43 rows have a NULL price. Verified two ways (LAG gap sum and calendar days minus row count).

**Decision:** in the combined table, forward-fill the last known price to every calendar day. Markets are closed on those days, so the last price is the true current price. 

## DQ Check 6 – Checking the true key of the holidays table

In [37]:
con.sql(""" DESCRIBE SELECT * FROM holidays_events""").df()
            

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,holiday_type,VARCHAR,YES,None,None,None
2,locale,VARCHAR,YES,None,None,None
3,locale_name,VARCHAR,YES,None,None,None
4,description,VARCHAR,YES,None,None,None
5,is_transferred,BOOLEAN,YES,None,None,None


** dates appear in holidays_events more than once

In [45]:
con.sql("""WITH duplicate_dates AS (
            SELECT 
            date,
            COUNT(*) AS dupl_date_count
            FROM holidays_events
            GROUP BY date
            HAVING COUNT(*) >1)
            SELECT
            COUNT(dupl_date_count) AS duplicate_dates_total_count
            FROM duplicate_dates
            """).df()
            

,duplicate_dates_total_count
0,31


** Is the combination date + locale_name unique

In [60]:
con.sql("""WITH date_locale_name_dupl AS (
            SELECT 
            date,
            locale_name,
            COUNT(*) AS dupl_date_count
            FROM holidays_events
            GROUP BY date,locale_name
            HAVING COUNT(*) >1)
            SELECT
            h.date,
            h.holiday_type,
            h.locale,
            h.locale_name,
            h.description,
            h.is_transferred,
            FROM date_locale_name_dupl d
            INNER JOIN holidays_events h
                ON h.date = d.date
                AND h.locale_name = d.locale_name
            """).df()
    

,date,holiday_type,locale,locale_name,description,is_transferred
0,2012-12-24,Bridge,National,Ecuador,Puente Navidad,False
1,2012-12-24,Additional,National,Ecuador,Navidad-1,False
2,2012-12-31,Bridge,National,Ecuador,Puente Primer dia del ano,False
3,2012-12-31,Additional,National,Ecuador,Primer dia del ano-1,False
4,2014-12-26,Bridge,National,Ecuador,Puente Navidad,False
5,2014-12-26,Additional,National,Ecuador,Navidad+1,False
6,2016-05-01,Holiday,National,Ecuador,Dia del Trabajo,False
7,2016-05-01,Event,National,Ecuador,Terremoto Manabi+15,False
8,2016-05-07,Additional,National,Ecuador,Dia de la Madre-1,False
9,2016-05-07,Event,National,Ecuador,Terremoto Manabi+21,False


** Key AS  date + locale_name + description

In [62]:
con.sql("""WITH duplicate_dates AS (
            SELECT 
            date,
            locale_name,
            description,
            COUNT(*) AS dupl_date_count
            FROM holidays_events
            GROUP BY date, locale_name, description
            HAVING COUNT(*) >1)
            SELECT
            COUNT(dupl_date_count) AS duplicate_dates_total_count
            FROM duplicate_dates
            """).df()

,duplicate_dates_total_count
0,0


**Finding:** `date` alone is not unique (31 dates repeat), and neither is `date + locale_name`. Repeats occur when several holidays or events fall on the same day in the same place, usually a main holiday plus an "Additional" day. On 2016-05-08, two events share the same type and differ only by description. True key: date + locale_name + description.

**Decision:** before joining to sales, aggregate holidays to one row per date + location as flags (holiday, event, additional day, etc.). Then join national on date, regional on date + state, local on date + city. This prevents fan-out.

## DQ Check 7 – Checking store coverage in the transactions table

** Store traffic (transactions) will be a demand sensing signal for every store and day. The planning team needs to know whether every store has a transaction record for every day.

For each store its first and last dates in transactions, and days within that range have no record. Sorted so the stores with the most missing days come first.

In [63]:
con.sql(""" DESCRIBE SELECT * FROM transactions""").df() 
            

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,store_nbr,INTEGER,YES,None,None,None
2,no_of_transactions,INTEGER,YES,None,None,None


In [82]:
con.sql("""SELECT
            store_nbr,
            MIN(date) AS first_txn_date,
            MAX(date) AS last_txn_date,
            Max(date) - Min(date) +1 - COUNT(*) AS no_of_days_no_record
            FROM transactions
            GROUP BY store_nbr
            ORDER BY no_of_days_no_record DESC""").df()
            

,store_nbr,first_txn_date,last_txn_date,no_of_days_no_record
0,18,2013-01-02,2017-08-15,121
1,24,2013-01-02,2017-08-15,110
2,25,2013-01-01,2017-08-15,73
3,12,2013-01-02,2017-08-15,71
4,14,2013-01-02,2017-08-15,49
5,30,2013-01-02,2017-08-15,32
6,43,2013-01-02,2017-08-15,15
7,17,2013-01-02,2017-08-15,13
8,7,2013-01-02,2017-08-15,12
9,10,2013-01-02,2017-08-15,12


In [81]:
con.sql("""SELECT * FROM transactions WHERE no_of_transactions IS NULL""").df()

,date,store_nbr,no_of_transactions


**Finding:** 45 stores cover 2013-01-02 to 2017-08-15 with ~9–11 missing days each, consistent with closure on Christmas and New Year's Day (likely true zeros, to be confirmed against sales). Six stores (18, 24, 25, 12, 14, 30) have 32–121 missing days, suggesting temporary closures or data gaps. Eight stores opened after 2013; store 52 opened 2017-04-20 (~4 months of history). All stores trade until 2017-08-15.

**Decisions:** (1) the store-day calendar starts at each store's own first date, never before opening; (2) exclude store 52 from model scope (too little history); (3) review outlier stores against sales when locking the scope; (4) missing transactions are not treated as 0 unless the store was closed.


## Checking date coverage across all tables

In [84]:
con.sql("""
        SELECT 'sales' AS date_range_table, 
        MIN (date) AS first_date,
        MAX (date) AS last_date
        FROM sales
        UNION ALL
        SELECT 'transactions' AS date_range_table,
        MIN (date) AS first_date,
        MAX (date) AS last_date
        FROM transactions
        UNION ALL
        SELECT 'oil' AS date_range_table, 
        MIN (date) AS first_date,
        MAX (date) AS last_date
        FROM oil
        UNION ALL
        SELECT 'holidays_events' AS date_range_table, 
        MIN (date) AS first_date,
        MAX (date) AS last_date
        FROM holidays_events""").df()

,date_range_table,first_date,last_date
0,sales,2013-01-01,2017-08-15
1,transactions,2013-01-01,2017-08-15
2,oil,2013-01-01,2017-08-31
3,holidays_events,2012-03-02,2017-12-26


 # | Check | Table | Result | Status | Decision |
|---|---|---|---|---|---|
| 1 | Duplicate business keys (date, store, item) | sales | 0 duplicates | Pass | Safe to join on business key |
| 2 | Unknown promotion flag | sales | 21.7M rows (17.3%), one clean block 2013-01-01 → 2014-03-31 | Known gap | No impact: model scope starts 2016 |
| 3 | Negative sales (net returns) | sales | 7,795 rows (0.006%); 0.013% of units | Immaterial | Keep raw; `unit_sales_clean` clips to 0 |
| 4 | Orphan items / stores | sales vs items, stores | 0 / 0 | Pass | All sales rows get master data and city |
| 5 | Oil gaps and NULL prices | oil | 529 of 1,704 days (31%): 486 no row + 43 NULL | Fixable | Forward-fill last known price (no leakage) |
| 6 | True key of holidays | holidays_events | Date not unique (31 dates repeat); key = date + locale_name + description | Fan-out risk | Aggregate to flags per date + location before joining |
| 7 | Missing store-days | transactions | Most stores ~10 days (holiday closures); 6 stores 32–121; 8 late openers; store 52 opened Apr 2017 | Known gap | Store calendar from own first date; exclude store 52; missing ≠ 0 |
| 8 | Date coverage across tables | all | sales = transactions; oil and holidays extend beyond | Pass | Holidays usable for the forecast day; oil and transactions lagged only |

**Data limitations (for README):**
- Sales are **net** units (sales − returns); gross sales and returns are not available separately.
- No **inventory/stockout** data, so a zero can mean no demand or an empty shelf.
- Oil and weather use **actuals**; production would use forecasts (oil futures, weather forecasts) and train on point-in-time history.    

In [2]:
con.close()